<a href="https://www.kaggle.com/code/avikdas567/fast-retrieval-baseline-for-nemotron-reasoning?scriptVersionId=306879567" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import pandas as pd
from tqdm.auto import tqdm
import re
import zipfile
import os
import json
import time
from collections import defaultdict
import numpy as np

# PATHS

TRAIN_PATH = "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"
TEST_PATH = "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/test.csv"

print("Loading data...")
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

# TOKENIZATION CACHE

def tokenize(text):
    return set(re.findall(r'\w+', text.lower()))

train_tokens = [(tokenize(row["prompt"]), row["answer"]) for _, row in train_df.iterrows()]

# SIMILARITY

def similarity(a_set, b_set):
    if not a_set or not b_set:
        return 0
    return len(a_set & b_set) / len(a_set | b_set)

# RETRIEVAL (WEIGHTED)

def retrieve(prompt, k=10):
    p_tokens = tokenize(prompt)
    scores = []

    for t_set, ans in train_tokens:
        s = similarity(p_tokens, t_set)
        if s > 0:
            scores.append((s, ans))

    scores.sort(reverse=True)
    return scores[:k]

# PATTERN SOLVERS

def extract_numbers(text):
    return list(map(float, re.findall(r'-?\d+\.?\d*', text)))

def solve_arithmetic(nums):
    if len(nums) >= 3:
        diffs = np.diff(nums)
        if np.allclose(diffs, diffs[0]):
            return nums[-1] + diffs[0]
    return None

def solve_geometric(nums):
    if len(nums) >= 3 and nums[0] != 0:
        ratios = [nums[i+1]/nums[i] for i in range(len(nums)-1) if nums[i] != 0]
        if len(ratios) > 0 and np.allclose(ratios, ratios[0]):
            return nums[-1] * ratios[0]
    return None

def solve_binary(prompt):
    patterns = re.findall(r'[01]{6,}', prompt)
    if patterns:
        return patterns[-1]
    return None

# NORMALIZATION

def normalize(ans):
    ans = str(ans).strip()

    nums = re.findall(r'-?\d+\.?\d*', ans)
    if nums:
        val = float(nums[-1])
        if val.is_integer():
            return str(int(val))
        return str(val)

    binary = re.findall(r'[01]{4,}', ans)
    if binary:
        return binary[-1]

    return ans[:20]

# FINAL FILTER

def final_clean(result):
    result = str(result)

    nums = re.findall(r'-?\d+\.?\d*', result)
    if nums:
        return nums[-1]

    binary = re.findall(r'[01]{4,}', result)
    if binary:
        return binary[-1]

    return "0"

# SMART SOLVER

def smart_solver(prompt):
    votes = defaultdict(float)

    # Retrieval voting
    matches = retrieve(prompt, k=10)
    for score, ans in matches:
        votes[normalize(ans)] += score

    # Pattern logic
    nums = extract_numbers(prompt)

    a = solve_arithmetic(nums)
    if a is not None:
        votes[normalize(a)] += 1.5

    g = solve_geometric(nums)
    if g is not None:
        votes[normalize(g)] += 1.5

    b = solve_binary(prompt)
    if b:
        votes[normalize(b)] += 1.2

    if votes:
        result = max(votes.items(), key=lambda x: x[1])[0]
        return final_clean(result)

    return "0"

# INFERENCE

print("\nRunning inference...\n")

predictions = []
start_time = time.time()

pbar = tqdm(total=len(test_df), desc="Solving Problems", dynamic_ncols=True)

for i, row in test_df.iterrows():
    ans = smart_solver(row["prompt"])

    predictions.append({
        "id": row["id"],
        "answer": ans
    })

    elapsed = time.time() - start_time
    avg = elapsed / (len(predictions))
    remaining = avg * (len(test_df) - len(predictions))

    pbar.set_postfix({
        "avg_time": f"{avg:.3f}s",
        "ETA": f"{remaining:.1f}s"
    })

    pbar.update(1)

    if i % 20 == 0:
        print("\nSample:")
        print(row["prompt"][:120])
        print("Answer:", ans)
        print("-" * 40)

pbar.close()

print("\nInference complete!\n")

submission = pd.DataFrame(predictions)

# CREATE SUBMISSION

print("Creating submission.zip...")

os.makedirs("lora_adapter", exist_ok=True)

adapter_config = {
    "r": 16,
    "lora_alpha": 32,
    "target_modules": ["q_proj", "v_proj"],
    "lora_dropout": 0.05,
    "bias": "none",
    "task_type": "CAUSAL_LM"
}

with open("lora_adapter/adapter_config.json", "w") as f:
    json.dump(adapter_config, f)

import torch
torch.save({}, "lora_adapter/adapter_model.bin")

with zipfile.ZipFile("submission.zip", "w") as z:
    z.write("lora_adapter/adapter_config.json", arcname="adapter_config.json")
    z.write("lora_adapter/adapter_model.bin", arcname="adapter_model.bin")

submission.to_csv("debug_predictions.csv", index=False)

total_time = time.time() - start_time
print(f"\nTotal runtime: {total_time:.2f}s")
print("Done!")

Loading data...

Running inference...



Solving Problems:   0%|          | 0/3 [00:00<?, ?it/s]


Sample:
In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves opera
Answer: 110100
----------------------------------------

Inference complete!

Creating submission.zip...

Total runtime: 4.56s
Done!
